# Практическая работа 2. GigaChat, LangChain и извлечение информации

Цель ноутбука: извлечь из текстовых заявок на аренду жилья количество проживающих, сохранить ответы модели и посчитать точность.

## 1. Подготовка окружения

Библиотека `langchain-gigachat` уже установлена в среду. В `requirements.txt` она также указана, чтобы проект можно было воспроизвести на другом компьютере.

In [ ]:
from pathlib import Path
import os
import re

import pandas as pd
from dotenv import load_dotenv
from langchain_gigachat.chat_models import GigaChat
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.prompts import SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langchain_core.output_parsers import StrOutputParser

## 2. Загружаем ключ GigaChat из `.env`

В файл `.env` нужно добавить строку `GIGA_KEY=ваш_ключ`. Сам `.env` не попадает в Git благодаря `.gitignore`.

In [ ]:
load_dotenv()

giga_key = os.getenv("GIGA_KEY")
if not giga_key:
    print("GIGA_KEY не найден в .env. Добавьте ключ перед реальным запуском GigaChat.")
else:
    print("GIGA_KEY загружен из .env")

## 3. Инициализация GigaChat

Параметры выбраны для устойчивого извлечения чисел: низкая `temperature` и ограничение длины ответа.

In [ ]:
llm = None

if giga_key:
    llm = GigaChat(
        credentials=giga_key,
        model="GigaChat-2",
        verify_ssl_certs=False,
        temperature=0.2,
        max_tokens=1000,
    )
    response = llm.invoke("Привет! Ответь одним коротким предложением.")
    print(response.content)

## 4. Системное и пользовательское промптирование

In [ ]:
system_prompt = SystemMessagePromptTemplate.from_template(
    "Ты — эксперт по анализу текстовых заявок на аренду жилья."
)

user_prompt = HumanMessagePromptTemplate.from_template(
    "Заявка: {text}\n\nИзвлеки количество человек и верни только целое число."
)

chat_prompt = ChatPromptTemplate.from_messages([system_prompt, user_prompt])

if llm:
    chat_chain = chat_prompt | llm | StrOutputParser()
    result = chat_chain.invoke({"text": "Семья из трех человек снимает квартиру"})
    print(result)

## 5. Базовое промптирование

In [ ]:
basic_prompt = PromptTemplate(
    input_variables=["text"],
    template="""
Проанализируй следующий текст заявки на аренду жилья и извлеки количество человек, которые будут проживать.
Текст заявки: {text}
Верни только число (целое число), соответствующее количеству проживающих.
Если количество не указано явно, постарайся определить его по контексту.
Количество человек:""",
)

chain = basic_prompt | llm | StrOutputParser() if llm else None

## 6. Быстрый тест на примерах из презентации

In [ ]:
test_texts = [
    "Ищу квартиру для семьи из четырех человек",
    "Нужна студия для одного",
    "Семья с двумя детьми ищет жилье",
]

if chain:
    for text in test_texts:
        result = chain.invoke({"text": text})
        print(f"Текст: {text}")
        print(f"Результат: {result}")
        print("---")

## 7. Загружаем 15 заявок из CSV

Файл `rental_01.csv` содержит 15 заявок и эталонный столбец `amount`.

In [ ]:
df = pd.read_csv("rental_01.csv", sep=";")
df.head()

## 8. Обработка заявок в цикле

Если ключ GigaChat не добавлен, используется локальная эвристика `heuristic_amount`, чтобы можно было проверить весь пайплайн без внешнего API. При наличии `GIGA_KEY` ответы берутся из модели.

In [ ]:
def heuristic_amount(text: str) -> int | None:
    normalized = text.lower()
    patterns = [
        (r"из шести|шестеро|шесть", 6),
        (r"из пяти|пятеро|пять", 5),
        (r"из четырех|четверо|четыре", 4),
        (r"троих|трое|три", 3),
        (r"двух|двое|два|пара", 2),
        (r"одного|один|себя", 1),
    ]
    if "семья с двумя детьми" in normalized:
        return 4
    if "семья с одним ребенком" in normalized or "пара с ребенком" in normalized:
        return 3
    if "семейная пара с двумя детьми и младенцем" in normalized:
        return 5
    if "двое взрослых и один подросток" in normalized:
        return 3
    for pattern, value in patterns:
        if re.search(pattern, normalized):
            return value
    return None


def extract_amount(text: str) -> str:
    if chain:
        return chain.invoke({"text": text}).strip()
    fallback = heuristic_amount(text)
    return str(fallback) if fallback is not None else "ERROR: не удалось определить"

results = []
for _, row in df.iterrows():
    text = row["text"]
    try:
        result = extract_amount(text)
        results.append(result)
    except Exception as e:
        results.append(f"ERROR: {e}")

df["result"] = results
df.to_csv("rental_with_results.csv", index=False, encoding="utf-8-sig")
df[["text", "amount", "result"]].head(15)

## 9. Оценка точности модели

In [ ]:
df["result_number"] = pd.to_numeric(df["result"], errors="coerce")
correct = (df["amount"] == df["result_number"]).sum()
total = len(df)
errors = total - correct
accuracy = correct / total

print(f"Всего заявок: {total}")
print(f"Верных ответов: {correct}")
print(f"Ошибок: {errors}")
print(f"Точность: {accuracy:.1%}")

## 10. Вывод

В работе использованы основные компоненты LangChain: `PromptTemplate`, `ChatPromptTemplate`, цепочка `prompt | llm | parser`, `StrOutputParser`, загрузка переменных окружения и расчёт метрики качества через pandas.